# Desafio A - Compras públicas e diferenças regionais 
---

## Problema
- A distribuiçao das contrataçoes públicas apresenta difereças a entre as Unidade da Federação?

## Estrategia de resolução 

- Vamos criar um mapa de calor com um filtro de anos, de 2020 a 2026. Será possível visualizar as contratações de cada ano e também o total acumulado do período.

O mapa vai mostrar todos os países e a quantidade de contratações realizadas. Quanto mais forte a cor, maior o número de contratações; quanto mais fraca, menor a quantidade. Assim, fica fácil identificar quais países mais contratam ao longo dos anos.

## Perguntas 
- Quantidade de contratação aentre os estados
- Estados Com mais contratações 
- Estados com menos contratações 
- Estados com maior concetração em diferentes modalidades 
- Os vaalores gastos por cada estaçao por ano analisado + somativo geral nacional 

## Análises 
- Frequencias 
- Mediana 
- Média 
- Variação 
- Maior 
- Minimo 

## Filtros utilizados da API
- Consultar Contratacoes 
- Consultar Itens
- Consualtar resultados itens contratacoes

## Bibliotecas

Utilizaremos principalmente:

- `requests` para realizar as requisições HTTP;
- `pandas` para organizar e analisar os dados;
- `matplotlib` para visualizações simples.

In [1]:
%pip install requests pandas matplotlib -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import time

## URLs ultilizadas:

A API do Compras.gov.br possui diferentes módulos.

Alguns exemplos:

```text
Catálogo de materiais
Catálogo de serviços
Pesquisa de preços
Contratações
Atas de Registro de Preços
Contratos
Fornecedores
```

Nesse projeto ultilizaremos o modulo de contratações, fazendo uso dos seguintes endpoints:

```text
GET /modulo-contratacoes/1_consultarContratacoes_PNCP_14133
GET //modulo-contratacoes/2_consultarItensContratacoes_PNCP_14133
GET //modulo-contratacoes/3_consultarResultadoItensContratacoes_PNCP_14133
```

Esse serviço consulta contratações realizadas no contexto da Lei nº 14.133/2021. Esta Lei estabelece normas gerais de licitação e contratação para as Administrações Públicas diretas, autárquicas e fundacionais da União, dos Estados, do Distrito Federal e dos Municípios

A URL base será:

In [3]:
base_url = "https://dadosabertos.compras.gov.br"

endpoint_contratacoes = "/modulo-contratacoes/1_consultarContratacoes_PNCP_14133"
endpoint_itens = "/modulo-contratacoes/2_consultarItensContratacoes_PNCP_14133"
endpoint_resultados_itens = "/modulo-contratacoes/3_consultarResultadoItensContratacoes_PNCP_14133"


## Parâmetros da consulta

O endpoint permite utilizar parâmetros para restringir a consulta.

Nesta atividade utilizaremos:

- `anos`: anos dos registros;
- `tamanhoPagina`: número de registros por página;
- `maximoPaginas`: maxímo de registros por página;
- `codigoModalidade`: código da modalidade;



In [4]:
anos = list(range(2020, 2027))          
modalidade = 6                         
tamanho_paginas = 50                 
max_paginas_ano = 20              

parametros = {
    "ano": anos,
    "tamanhoPagina": tamanho_paginas,
    "maximoPaginas": max_paginas_ano,
    "codigoModalidade": modalidade,
}

parametros

{'ano': [2020, 2021, 2022, 2023, 2024, 2025, 2026],
 'tamanhoPagina': 50,
 'maximoPaginas': 20,
 'codigoModalidade': 6}

## Consulta simples dos dados

Antes da coleta completa (2020–2026), fizemos uma consulta pequena para verificar:

- o endpoint estava disponível?
- os campos necessários (UF, valor, modalidade) estão presentes?
- o `codigoModalidade` escolhido retorna registros?

In [5]:
def extrair_informacoes(dados):
    return dados["resultado"]

params_dados = {
    "pagina": 1,
    "tamanhoPagina": 10,
    "dataPublicacaoPncpInicial": "2025-05-01",
    "dataPublicacaoPncpFinal": "2025-05-15",
    "codigoModalidade": modalidade,
}

resposta = requests.get(base_url + endpoint_contratacoes, params=params_dados)

print("Status HTTP:", resposta.status_code)
print("URL consultada:", resposta.url)

registros = extrair_informacoes(resposta.json())
df_simples = pd.json_normalize(registros)

print("Registros encontrados:", len(df_simples))
df_simples.head()

Status HTTP: 200
URL consultada: https://dadosabertos.compras.gov.br/modulo-contratacoes/1_consultarContratacoes_PNCP_14133?pagina=1&tamanhoPagina=10&dataPublicacaoPncpInicial=2025-05-01&dataPublicacaoPncpFinal=2025-05-15&codigoModalidade=6
Registros encontrados: 10


,idCompra,numeroControlePNCP,anoCompraPncp,sequencialCompraPncp,orgaoEntidadeCnpj,orgaoSubrogadoCnpj,codigoOrgao,orgaoEntidadeRazaoSocial,orgaoSubrogadoRazaoSocial,orgaoEntidadeEsferaId,...,tipoInstrumentoConvocatorioNome,modoDisputaNomePncp,valorTotalEstimado,valorTotalHomologado,dataInclusaoPncp,dataAtualizacaoPncp,dataPublicacaoPncp,dataAberturaPropostaPncp,dataEncerramentoPropostaPncp,contratacaoExcluida
0,78251006900072025,00394502000144-1-003952/2025,2025,3952,00394502000144,None,46041,COMANDO DA MARINHA,None,F,...,Aviso de Contratação Direta,Dispensa Com Disputa,34103.12,NaN,2025-05-01T08:48:39,2025-05-01T08:48:39,2025-05-01T08:48:39,2025-05-01T08:48:36,2025-05-08T07:59:59,False
1,92826706900152025,14846532000159-1-000021/2025,2025,21,14846532000159,None,67124,CONSELHO DE ARQUITETURA E URBANISMO DO AMAPA,None,F,...,Aviso de Contratação Direta,Dispensa Com Disputa,4100.00,1500.0,2025-05-01T11:08:48,2025-05-01T11:08:48,2025-05-01T11:08:48,2025-05-01T11:08:47,2025-05-08T07:59:59,False
2,78882006900212025,00394502000144-1-003953/2025,2025,3953,00394502000144,None,46041,COMANDO DA MARINHA,None,F,...,Aviso de Contratação Direta,Dispensa Com Disputa,7451.86,NaN,2025-05-01T11:24:21,2025-05-01T11:24:21,2025-05-01T11:24:21,2025-05-01T11:24:19,2025-05-08T07:59:59,False
3,78790006900502025,00394502000144-1-003954/2025,2025,3954,00394502000144,None,46041,COMANDO DA MARINHA,None,F,...,Aviso de Contratação Direta,Dispensa Com Disputa,4667.26,4275.7,2025-05-01T12:12:43,2025-05-01T12:12:43,2025-05-01T12:12:43,2025-05-01T12:12:42,2025-05-08T07:59:59,False
4,16000606900072025,00394452000103-1-009791/2025,2025,9791,00394452000103,None,44611,COMANDO DO EXERCITO,None,F,...,Aviso de Contratação Direta,Dispensa Com Disputa,76274.86,63987.0,2025-05-01T12:17:43,2025-05-01T12:17:43,2025-05-01T12:17:43,2025-05-01T12:17:42,2025-05-13T08:59:59,False


In [6]:
colunas = [
    "idCompra",
    "numeroCompra",
    "anoCompraPncp",
    "modalidadeNome",
    "modalidadeIdPncp",
    "unidadeOrgaoNomeUnidade",
    "orgaoEntidadeRazaoSocial",
    "objetoCompra",
    "valorTotalEstimado",
    "valorTotalHomologado",
    "dataPublicacaoPncp",
    "srp",
]

colunas_existentes = []

for coluna in colunas:
    if coluna in df_simples.columns:
        colunas_existentes.append(coluna)

print("Colunas retornadas pela API:")
print(df_simples.columns.tolist())

print("\nColunas de interesse encontradas:")
print(colunas_existentes)

print("\nColuna da UF:", "unidadeOrgaoUfSigla")

Colunas retornadas pela API:
['idCompra', 'numeroControlePNCP', 'anoCompraPncp', 'sequencialCompraPncp', 'orgaoEntidadeCnpj', 'orgaoSubrogadoCnpj', 'codigoOrgao', 'orgaoEntidadeRazaoSocial', 'orgaoSubrogadoRazaoSocial', 'orgaoEntidadeEsferaId', 'orgaoSubrogadoEsferaId', 'orgaoEntidadePoderId', 'orgaoSubrogadoPoderId', 'unidadeOrgaoCodigoUnidade', 'unidadeSubrogadaCodigoUnidade', 'unidadeOrgaoNomeUnidade', 'unidadeSubrogadaNomeUnidade', 'unidadeOrgaoUfSigla', 'unidadeSubrogadaUfSigla', 'unidadeOrgaoMunicipioNome', 'unidadeSubrogadaMunicipioNome', 'unidadeOrgaoCodigoIbge', 'unidadeSubrogadaCodigoIbge', 'numeroCompra', 'modalidadeIdPncp', 'codigoModalidade', 'modalidadeNome', 'srp', 'modoDisputaIdPncp', 'codigoModoDisputa', 'amparoLegalCodigoPncp', 'amparoLegalNome', 'amparoLegalDescricao', 'informacaoComplementar', 'processo', 'objetoCompra', 'existeResultado', 'orcamentoSigilosoCodigo', 'orcamentoSigilosoDescricao', 'situacaoCompraIdPncp', 'situacaoCompraNomePncp', 'tipoInstrumentoConvo

## 9. Coleta dos dados

* Essa função irá coletar, dados de um determinado ano, de acordo com o `max`.

* seguindo as boas práticas da aula: paginação controlada, `timeout`, interrupção quando a página vier vazia.

In [7]:
def coletar_ano(ano):
    data_inicial = f"{ano}-01-01"
    data_final = f"{ano}-12-31"

    todos_registros = []

    for pagina in range(1, max_paginas_ano + 1):

        params = {
            "pagina": pagina,
            "tamanhoPagina": tamanho_paginas,
            "dataPublicacaoPncpInicial": data_inicial,
            "dataPublicacaoPncpFinal": data_final,
            "codigoModalidade": modalidade,
        }

        resposta = requests.get(
            base_url + endpoint_contratacoes,
            params=params,
            timeout=60
        )

        print(f"Ano {ano} - Página {pagina} - Status: {resposta.status_code}")

        if resposta.status_code != 200:
            break

        registros = extrair_informacoes(resposta.json())

        print(f"Registros encontrados: {len(registros)}")

        if len(registros) == 0:
            break

        todos_registros.extend(registros)

        if len(registros) < tamanho_paginas:
            break

        time.sleep(0.3)

    df_ano = pd.json_normalize(todos_registros)

    if not df_ano.empty:
        df_ano["anoConsulta"] = ano

    return df_ano

In [8]:
dataframes_por_ano = []

for ano in anos:
    print(f"Coletando ano {ano}...")

    df_ano = coletar_ano(ano)

    print(f"Registros coletados em {ano}: {len(df_ano)}")

    dataframes_por_ano.append(df_ano)

df_infos = pd.concat(dataframes_por_ano, ignore_index=True)

print("Total de registros:", len(df_infos))

df_infos.head()

Coletando ano 2020...


Ano 2020 - Página 1 - Status: 200
Registros encontrados: 0
Registros coletados em 2020: 0
Coletando ano 2021...
Ano 2021 - Página 1 - Status: 200
Registros encontrados: 50
Ano 2021 - Página 2 - Status: 200
Registros encontrados: 50
Ano 2021 - Página 3 - Status: 200
Registros encontrados: 50
Ano 2021 - Página 4 - Status: 200
Registros encontrados: 50
Ano 2021 - Página 5 - Status: 200
Registros encontrados: 50
Ano 2021 - Página 6 - Status: 200
Registros encontrados: 50
Ano 2021 - Página 7 - Status: 200
Registros encontrados: 50
Ano 2021 - Página 8 - Status: 200
Registros encontrados: 50
Ano 2021 - Página 9 - Status: 200
Registros encontrados: 50
Ano 2021 - Página 10 - Status: 200
Registros encontrados: 50
Ano 2021 - Página 11 - Status: 200
Registros encontrados: 50
Ano 2021 - Página 12 - Status: 200
Registros encontrados: 50
Ano 2021 - Página 13 - Status: 200
Registros encontrados: 50
Ano 2021 - Página 14 - Status: 200
Registros encontrados: 50
Ano 2021 - Página 15 - Status: 200
Registro

,idCompra,numeroControlePNCP,anoCompraPncp,sequencialCompraPncp,orgaoEntidadeCnpj,orgaoSubrogadoCnpj,codigoOrgao,orgaoEntidadeRazaoSocial,orgaoSubrogadoRazaoSocial,orgaoEntidadeEsferaId,...,modoDisputaNomePncp,valorTotalEstimado,valorTotalHomologado,dataInclusaoPncp,dataAtualizacaoPncp,dataPublicacaoPncp,dataAberturaPropostaPncp,dataEncerramentoPropostaPncp,contratacaoExcluida,anoConsulta
0,78333006000012021,00394502000144-1-000001/2021,2021,1,00394502000144,None,46041,COMANDO DA MARINHA,None,F,...,Não se aplica,49506.60,NaN,2021-08-10T07:54:24,2021-08-10T07:54:24,2021-08-10T07:54:24,2021-08-10T08:00:00,2021-08-13T14:00:00,False,2021
1,15327906000042021,17217985000104-1-000001/2021,2021,1,17217985000104,None,16209,UNIVERSIDADE FEDERAL DE MINAS GERAIS,None,F,...,Não se aplica,23147.09,10684.52,2021-08-10T07:54:38,2021-08-10T07:54:38,2021-08-10T07:54:38,2021-08-10T08:00:00,2021-08-13T15:00:00,False,2021
2,15329206000052021,17217985000104-1-000002/2021,2021,2,17217985000104,None,16209,UNIVERSIDADE FEDERAL DE MINAS GERAIS,None,F,...,Não se aplica,43928.08,29959.13,2021-08-10T07:54:39,2021-08-10T07:54:39,2021-08-10T07:54:39,2021-08-10T08:00:00,2021-08-13T14:00:00,False,2021
3,15327906000072021,17217985000104-1-000003/2021,2021,3,17217985000104,None,16209,UNIVERSIDADE FEDERAL DE MINAS GERAIS,None,F,...,Não se aplica,0.00,385.00,2021-08-10T07:54:41,2021-08-10T07:54:41,2021-08-10T07:54:41,2021-08-10T08:00:00,2021-08-13T15:00:00,False,2021
4,15667806000112021,35840659000130-1-000001/2021,2021,1,35840659000130,None,81237,UNIVERSIDADE FEDERAL DE JATAI,None,F,...,Não se aplica,59188.98,NaN,2021-08-10T07:54:42,2021-08-10T07:54:42,2021-08-10T07:54:42,2021-08-10T08:00:00,2021-08-13T16:00:00,False,2021


## Dataframe com as informações gerais

In [9]:
coluna_uf = "unidadeOrgaoUfSigla"

print("Coluna de UF usada:", coluna_uf)

colunas_finais = colunas + ["anoConsulta", coluna_uf]

df = df_infos[colunas_finais].copy()

df = df.rename(columns={
    "unidadeOrgaoUfSigla": "uf"
})

print("Dimensão do DataFrame:", df.shape)

df.head()

Coluna de UF usada: unidadeOrgaoUfSigla
Dimensão do DataFrame: (6000, 14)


,idCompra,numeroCompra,anoCompraPncp,modalidadeNome,modalidadeIdPncp,unidadeOrgaoNomeUnidade,orgaoEntidadeRazaoSocial,objetoCompra,valorTotalEstimado,valorTotalHomologado,dataPublicacaoPncp,srp,anoConsulta,uf
0,78333006000012021,00001,2021,Dispensa,8,CAPITANIA DOS PORTOS DA PARAÍBA,COMANDO DA MARINHA,Contratação de Empresa especializada em Obras/...,49506.60,NaN,2021-08-10T07:54:24,False,2021,PB
1,15327906000042021,00004,2021,Dispensa,8,ESCOLA DE ENFERMAGEM/UFMG,UNIVERSIDADE FEDERAL DE MINAS GERAIS,Aquisição de materiais laboratoriais e hospita...,23147.09,10684.52,2021-08-10T07:54:38,False,2021,MG
2,15329206000052021,00005,2021,Dispensa,8,INSTITUTO DE CIENCIAS EXATAS/UFMG,UNIVERSIDADE FEDERAL DE MINAS GERAIS,Aquisição de material elétrico,43928.08,29959.13,2021-08-10T07:54:39,False,2021,MG
3,15327906000072021,00007,2021,Dispensa,8,ESCOLA DE ENFERMAGEM/UFMG,UNIVERSIDADE FEDERAL DE MINAS GERAIS,Aquisição de açúcar cristal (pacote com 5 quil...,0.00,385.00,2021-08-10T07:54:41,False,2021,MG
4,15667806000112021,00011,2021,Dispensa,8,UNIVERSIDADE FEDERAL DE JATAI,UNIVERSIDADE FEDERAL DE JATAI,Serviço de instalação de um (1) gerador de 350...,59188.98,NaN,2021-08-10T07:54:42,False,2021,GO


## Verificação de informações faltantes e duplicados

In [10]:
print("Linhas duplicadas:", df.duplicated().sum())

print("IDs de compra duplicados:", df["idCompra"].duplicated().sum())

print("\nValores ausentes por coluna (%):")

percentual_ausentes = df.isna().mean() * 100

print(percentual_ausentes)

Linhas duplicadas: 0
IDs de compra duplicados: 222

Valores ausentes por coluna (%):
idCompra                     0.000000
numeroCompra                 0.000000
anoCompraPncp                0.000000
modalidadeNome               0.000000
modalidadeIdPncp             0.000000
unidadeOrgaoNomeUnidade      0.000000
orgaoEntidadeRazaoSocial     0.000000
objetoCompra                 0.000000
valorTotalEstimado           0.000000
valorTotalHomologado        27.166667
dataPublicacaoPncp           0.000000
srp                          0.000000
anoConsulta                  0.000000
uf                           0.000000
dtype: float64


## Limpeza dos dados

In [11]:
df_limpo = df.copy()

# Remove duplicados
antes = len(df_limpo)

df_limpo = df_limpo.drop_duplicates(subset="idCompra")

print("Duplicados removidos:", antes - len(df_limpo))

# Converte os valores para números
df_limpo["valorTotalEstimado"] = pd.to_numeric(
    df_limpo["valorTotalEstimado"],
    errors="coerce"
)

df_limpo["valorTotalHomologado"] = pd.to_numeric(
    df_limpo["valorTotalHomologado"],
    errors="coerce"
)

# Converte a data
df_limpo["dataPublicacaoPncp"] = pd.to_datetime(
    df_limpo["dataPublicacaoPncp"],
    errors="coerce"
)

# Remove registros sem UF
antes = len(df_limpo)

df_limpo = df_limpo.dropna(subset=["uf"])

print("Registros sem UF removidos:", antes - len(df_limpo))

print("\nDimensão final:", df_limpo.shape)

df_limpo.head()

Duplicados removidos: 222
Registros sem UF removidos: 0

Dimensão final: (5778, 14)


,idCompra,numeroCompra,anoCompraPncp,modalidadeNome,modalidadeIdPncp,unidadeOrgaoNomeUnidade,orgaoEntidadeRazaoSocial,objetoCompra,valorTotalEstimado,valorTotalHomologado,dataPublicacaoPncp,srp,anoConsulta,uf
0,78333006000012021,00001,2021,Dispensa,8,CAPITANIA DOS PORTOS DA PARAÍBA,COMANDO DA MARINHA,Contratação de Empresa especializada em Obras/...,49506.60,NaN,2021-08-10 07:54:24,False,2021,PB
1,15327906000042021,00004,2021,Dispensa,8,ESCOLA DE ENFERMAGEM/UFMG,UNIVERSIDADE FEDERAL DE MINAS GERAIS,Aquisição de materiais laboratoriais e hospita...,23147.09,10684.52,2021-08-10 07:54:38,False,2021,MG
2,15329206000052021,00005,2021,Dispensa,8,INSTITUTO DE CIENCIAS EXATAS/UFMG,UNIVERSIDADE FEDERAL DE MINAS GERAIS,Aquisição de material elétrico,43928.08,29959.13,2021-08-10 07:54:39,False,2021,MG
3,15327906000072021,00007,2021,Dispensa,8,ESCOLA DE ENFERMAGEM/UFMG,UNIVERSIDADE FEDERAL DE MINAS GERAIS,Aquisição de açúcar cristal (pacote com 5 quil...,0.00,385.00,2021-08-10 07:54:41,False,2021,MG
4,15667806000112021,00011,2021,Dispensa,8,UNIVERSIDADE FEDERAL DE JATAI,UNIVERSIDADE FEDERAL DE JATAI,Serviço de instalação de um (1) gerador de 350...,59188.98,NaN,2021-08-10 07:54:42,False,2021,GO


## Estatística descritiva

In [12]:
valor = "valorTotalHomologado"

freq_uf = df_limpo["uf"].value_counts().rename("quantidade_contratacoes")

stats_valor_uf = df_limpo.groupby("uf")[valor].agg(
    total_gasto="sum",
    mediana="median",
    media="mean",
    desvio_padrao="std",
    minimo="min",
    maximo="max"
)

stats_valor_uf["coef_variacao"] = (
    stats_valor_uf["desvio_padrao"] / stats_valor_uf["media"]
)

estatisticas_uf = stats_valor_uf.copy()

estatisticas_uf["quantidade_contratacoes"] = freq_uf

estatisticas_uf = estatisticas_uf.sort_values(
    "quantidade_contratacoes",
    ascending=False
)

estatisticas_uf["total_gasto"] = estatisticas_uf["total_gasto"].apply(
    lambda x: f"R$ {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
)
estatisticas_uf["mediana"] = estatisticas_uf["mediana"].apply(
    lambda x: f" {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
)
estatisticas_uf["media"] = estatisticas_uf["media"].apply(
    lambda x: f" {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
)
estatisticas_uf["desvio_padrao"] = estatisticas_uf["desvio_padrao"].apply(
    lambda x: f" {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
)
estatisticas_uf["minimo"] = estatisticas_uf["minimo"].apply(
    lambda x: f" {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
)
estatisticas_uf["maximo"] = estatisticas_uf["maximo"].apply(
    lambda x: f" {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
)
estatisticas_uf

,total_gasto,mediana,media,desvio_padrao,minimo,maximo,coef_variacao,quantidade_contratacoes
uf,,,,,,,,
SP,"R$ 635.817.382,37","8.400,00","648.131,89","7.593.433,16","0,00","137.035.308,92",11.715877,1175
RJ,"R$ 173.068.767,76","7.282,37","294.836,06","2.691.621,61","0,00","35.091.700,00",9.129214,897
DF,"R$ 1.074.267.790,99","10.500,00","2.975.811,06","48.404.136,73","0,02","918.700.100,00",16.265864,471
RS,"R$ 48.291.885,62","3.468,18","158.854,89","1.444.406,45","35,40","21.548.888,74",9.092616,426
MG,"R$ 187.342.544,63","8.705,55","589.127,50","7.031.736,66","0,00","123.176.842,20",11.935849,417
PR,"R$ 8.651.768,95","6.500,00","41.198,90","147.260,40","109,99","1.676.830,14",3.574377,287
BA,"R$ 15.445.051,24","12.729,02","108.767,97","604.863,63","0,00","6.256.165,55",5.561046,205
GO,"R$ 9.085.817,43","18.101,80","76.998,45","285.242,45","2,00","2.148.000,00",3.704522,191
RO,"R$ 9.552.778,24","18.990,00","122.471,52","362.005,66","0,01","2.022.000,00",2.955836,189


## Mapa de calor por Unidade da Federação (Folium)

Como planejado na Estratégia de resolução, vamos construir um mapa de calor (choropleth) do Brasil,
colorindo cada estado pela **quantidade de contratações**, com um filtro para escolher o ano
(2020 a 2026) ou o **total acumulado do período**.

Usaremos:
- `folium` para desenhar o mapa;
- a malha geográfica oficial dos estados brasileiros, obtida na **API de Malhas do IBGE**;
- `ipywidgets` para criar o filtro (dropdown) de ano/total;
- os dados já coletados e limpos em `df_limpo`.


In [13]:
%pip install folium ipywidgets -q


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
import folium
from ipywidgets import interact, Dropdown
from IPython.display import display, Markdown


### Malha geográfica das UFs (fonte: IBGE)

A API de Malhas do IBGE devolve o contorno de cada estado em GeoJSON, identificando cada polígono
pelo código numérico do IBGE (ex.: `"35"` = São Paulo). Como nossos dados usam a sigla da UF
(`SP`, `RJ`, ...), fazemos essa tradução logo em seguida.


In [15]:
def buscar_malha_ufs():
    tentativas = [
        {
            "url": "https://servicodados.ibge.gov.br/api/v3/malhas/BR",
            "params": {"formato": "application/vnd.geo+json", "intrarregiao": "UF"},
        },
        {
            "url": "https://servicodados.ibge.gov.br/api/v2/malhas/BR",
            "params": {"formato": "application/vnd.geo+json", "resolucao": 2},
        },
        {
            "url": "https://servicodados.ibge.gov.br/api/v3/malhas/BR",
            "params": {"formato": "application/vnd.geo+json", "resolucao": 2},
        },
    ]

    for tentativa in tentativas:
        try:
            resp = requests.get(tentativa["url"], params=tentativa["params"], timeout=60)
            print(f"Tentando {resp.url} -> status {resp.status_code}")

            if resp.status_code == 200:
                return resp.json()
        except requests.exceptions.RequestException as erro:
            print(f"Falhou ao tentar {tentativa['url']}: {erro}")

    raise RuntimeError(
        "Não foi possível obter a malha de UFs em nenhum dos endpoints testados. "
        "Verifique sua conexão com a internet (ou proxy/firewall) e tente novamente."
    )


geojson_uf = buscar_malha_ufs()

print("\nEstados na malha:", len(geojson_uf["features"]))
print("Propriedades de exemplo:", geojson_uf["features"][0]["properties"])


Tentando https://servicodados.ibge.gov.br/api/v3/malhas/BR?formato=application%2Fvnd.geo%2Bjson&intrarregiao=UF -> status 404
Tentando https://servicodados.ibge.gov.br/api/v2/malhas/BR?formato=application%2Fvnd.geo%2Bjson&resolucao=2 -> status 200

Estados na malha: 27
Propriedades de exemplo: {'codarea': '11', 'centroide': [-62.84169849643017, -10.91332465228205]}


In [16]:
CHAVES_CANDIDATAS = ["codarea", "id", "CD_UF", "GEOCODIGO", "codigo_ibg", "cod_uf"]

# Tabela oficial de códigos do IBGE -> sigla da UF (a mesma usada em df_limpo["uf"])
codigo_para_uf = {
    "11": "RO", "12": "AC", "13": "AM", "14": "RR", "15": "PA", "16": "AP", "17": "TO",
    "21": "MA", "22": "PI", "23": "CE", "24": "RN", "25": "PB", "26": "PE", "27": "AL",
    "28": "SE", "29": "BA", "31": "MG", "32": "ES", "33": "RJ", "35": "SP",
    "41": "PR", "42": "SC", "43": "RS", "50": "MS", "51": "MT", "52": "GO", "53": "DF",
}

features = geojson_uf["features"]
propriedades_exemplo = features[0]["properties"]

# Descobre automaticamente qual propriedade do GeoJSON guarda o código da UF
chave_codigo = next((c for c in CHAVES_CANDIDATAS if c in propriedades_exemplo), None)

if chave_codigo is None:
    raise KeyError(
        "Não encontrei uma propriedade de código conhecida no GeoJSON. "
        f"Propriedades disponíveis: {list(propriedades_exemplo.keys())}"
    )

print("Propriedade usada como código da UF:", chave_codigo)

# Converte código -> sigla e já registra os códigos sem correspondência
ufs_sem_mapeamento = []
for feature in features:
    props = feature["properties"]
    codigo = str(props[chave_codigo])
    props["uf"] = codigo_para_uf.get(codigo, codigo)

    if codigo not in codigo_para_uf:
        ufs_sem_mapeamento.append(codigo)

if ufs_sem_mapeamento:
    print("Atenção, códigos sem sigla mapeada:", ufs_sem_mapeamento)
else:
    print("Todos os 27 códigos foram convertidos em sigla com sucesso.")

Propriedade usada como código da UF: codarea
Todos os 27 códigos foram convertidos em sigla com sucesso.


### Classificação por área (Saúde, Educação, etc.)

O PNCP **não** devolve um campo pronto de "área/setor" (saúde, educação...). Para estimar isso,
criamos uma classificação heurística: procuramos palavras-chave no texto de `objetoCompra`
(a descrição da contratação) e associamos cada registro à área mais provável.


In [17]:
import unicodedata


def normalizar(texto):
    if not isinstance(texto, str):
        return ""
    return unicodedata.normalize("NFKD", texto).encode("ASCII", "ignore").decode("ASCII").lower()


categorias_palavras_chave = {
    "Saúde": ["saude", "hospital", "medic", "clinica", "enferm", "upa ", "farmac", "odontolog", "ambulanc"],
    "Educação": ["educa", "escola", "ensino", "universidad", "creche", "professor", "pedagog", "didatic"],
    "Segurança/Defesa": ["seguranca", "policia", "militar", "bombeiro", "defesa", "penitenciari", "presidio"],
    "Infraestrutura/Obras": ["obra", "construcao", "pavimenta", "engenharia", "reforma", "rodovia", "predio"],
    "Tecnologia/TI": ["informatica", "software", "sistema", "computador", "tecnologia da informacao", "rede de dados", "servidor"],
    "Alimentação": ["alimenta", "merenda", "genero alimenticio", "refeicao", "hortifruti"],
    "Transporte": ["veiculo", "transporte", "combustivel", "frota", "pneu", "locacao de onibus"],
    "Meio Ambiente/Saneamento": ["ambiental", "saneamento", "residuo solido", "coleta de lixo", "agua e esgoto"],
    "Administração/Serviços Gerais": ["limpeza", "vigilancia", "material de expediente", "papelaria", "servicos gerais"],
}


def classificar_area(objeto):
    texto = normalizar(objeto)
    return next(
        (area for area, palavras in categorias_palavras_chave.items()
         if any(palavra in texto for palavra in palavras)),
        "Outros",
    )


df_limpo["area"] = df_limpo["objetoCompra"].apply(classificar_area)

print("Distribuição geral por área (2020-2026):")
print(df_limpo["area"].value_counts())

Distribuição geral por área (2020-2026):
area
Outros                           3537
Saúde                             606
Tecnologia/TI                     348
Infraestrutura/Obras              345
Educação                          273
Transporte                        242
Segurança/Defesa                  234
Administração/Serviços Gerais     112
Alimentação                        72
Meio Ambiente/Saneamento            9
Name: count, dtype: int64


### Quantidade de contratações por UF, por ano, e total do período


In [18]:
contagem_ano_uf = (
    df_limpo
    .groupby(["anoConsulta", "uf"])
    .size()
    .unstack(fill_value=0)
    .sort_index()
)

# Linha extra com o somativo geral do período (2020-2026) por UF
contagem_ano_uf.loc["Total"] = contagem_ano_uf.sum()

print("Total geral de contratações no período:", int(contagem_ano_uf.loc["Total"].sum()))
contagem_ano_uf


Total geral de contratações no período: 5778


uf,AC,AL,AM,AP,BA,CE,DF,ES,GO,MA,...,PR,RJ,RN,RO,RR,RS,SC,SE,SP,TO
anoConsulta,,,,,,,,,,,,,,,,,,,,,
2021,5,1,34,0,31,15,57,7,41,8,...,70,141,10,25,3,100,35,8,74,11
2022,8,1,23,1,42,14,96,31,38,7,...,23,219,22,62,2,83,29,5,90,5
2023,3,10,20,7,34,16,121,23,38,22,...,23,170,21,7,0,94,25,4,110,12
2024,3,16,6,3,47,28,93,14,41,23,...,69,165,25,23,5,72,24,1,173,16
2025,3,10,13,2,18,14,54,56,18,10,...,39,104,11,27,6,47,14,3,370,19
2026,11,19,14,5,33,27,50,13,15,13,...,63,98,17,45,11,30,17,9,358,7
Total,33,57,110,18,205,114,471,144,191,83,...,287,897,106,189,27,426,144,30,1175,70


In [19]:
def formatar_moeda(x):
    return f"R$ {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

quantidade_por_periodo = contagem_ano_uf.sum(axis=1)

valor_por_periodo = df_limpo.groupby("anoConsulta")["valorTotalHomologado"].sum()
valor_por_periodo.loc["Total"] = df_limpo["valorTotalHomologado"].sum()

def top_areas_periodo(periodo, top_n=2):
    d = df_limpo if periodo == "Total" else df_limpo[df_limpo["anoConsulta"] == periodo]
    return d[d["area"] != "Outros"]["area"].value_counts().head(top_n)

In [20]:
def filtrar_periodo(periodo):
    """Devolve df_limpo inteiro para 'Total' ou só as linhas do ano pedido."""
    return df_limpo if periodo == "Total" else df_limpo[df_limpo["anoConsulta"] == periodo]


def criar_mapa(periodo):
    df_mapa = contagem_ano_uf.loc[[periodo]].T.reset_index()
    df_mapa.columns = ["uf", "quantidade"]

    valor_uf_periodo = filtrar_periodo(periodo).groupby("uf")["valorTotalHomologado"].sum()
    df_mapa["valor"] = df_mapa["uf"].map(valor_uf_periodo).fillna(0)
    df_mapa["valor_fmt"] = df_mapa["valor"].apply(formatar_moeda)

    titulo_legenda = (
        "Quantidade de contratações - total 2020 a 2026"
        if periodo == "Total"
        else f"Quantidade de contratações em {periodo}"
    )

    # tiles="OpenStreetMap" -> mapa-base gratuito, sem exigir chave de API
    m = folium.Map(location=[-14.2, -51.9], zoom_start=4, tiles="OpenStreetMap")

    folium.Choropleth(
        geo_data=geojson_uf,
        data=df_mapa,
        columns=["uf", "quantidade"],
        key_on="feature.properties.uf",
        fill_color="YlOrRd",
        fill_opacity=0.85,
        line_opacity=0.4,
        nan_fill_color="white",
        legend_name=titulo_legenda,
    ).add_to(m)

    # Adiciona quantidade e valor ao GeoJSON para aparecer no tooltip (ao passar o mouse)
    dados_uf = df_mapa.set_index("uf")
    quantidade_uf = dados_uf["quantidade"].to_dict()
    valor_fmt_uf = dados_uf["valor_fmt"].to_dict()

    for feature in geojson_uf["features"]:
        props = feature["properties"]
        props["quantidade"] = int(quantidade_uf.get(props["uf"], 0))
        props["valor_fmt"] = valor_fmt_uf.get(props["uf"], formatar_moeda(0))

    folium.GeoJson(
        geojson_uf,
        style_function=lambda x: {"fillOpacity": 0, "weight": 0},
        tooltip=folium.GeoJsonTooltip(
            fields=["uf", "quantidade", "valor_fmt"],
            aliases=["UF:", "Contratações:", "Valor homologado:"],
        ),
    ).add_to(m)

    return m


def resumo_periodo(periodo):
    quantidade = int(quantidade_por_periodo.loc[periodo])
    valor = valor_por_periodo.loc[periodo]
    principais_areas = top_areas_periodo(periodo, top_n=2)

    linhas_areas = "\n".join(
        f"{i}. **{area}** — {qtd} contratações"
        for i, (area, qtd) in enumerate(principais_areas.items(), start=1)
    )

    titulo_periodo = "período completo (2020 a 2026)" if periodo == "Total" else f"ano de {periodo}"

    return Markdown(
        f"**Resumo — {titulo_periodo}**\n\n"
        f"- Quantidade de contratações: **{quantidade}**\n"
        f"- Valor total homologado: **{formatar_moeda(valor)}**\n"
        f"- Áreas mais contratadas:\n{linhas_areas}\n"
    )


def mostrar_mapa(periodo):
    display(criar_mapa(periodo))
    display(resumo_periodo(periodo))


opcoes_periodo = list(contagem_ano_uf.index)

interact(
    mostrar_mapa,
    periodo=Dropdown(options=opcoes_periodo, value="Total", description="Período:"),
);

interactive(children=(Dropdown(description='Período:', index=6, options=(2021, 2022, 2023, 2024, 2025, 2026, '…

### Mapa interativo com filtro de ano

Escolha um ano no menu para ver a distribuição das contratações naquele ano, ou selecione
**Total** para ver o acumulado de 2020 a 2026. Quanto mais escura a cor, maior a quantidade
de contratações no estado.


In [21]:
def criar_mapa(periodo):
    df_mapa = contagem_ano_uf.loc[[periodo]].T.reset_index()
    df_mapa.columns = ["uf", "quantidade"]

    if periodo == "Total":
        valor_uf_periodo = df_limpo.groupby("uf")["valorTotalHomologado"].sum()
    else:
        valor_uf_periodo = (
            df_limpo[df_limpo["anoConsulta"] == periodo]
            .groupby("uf")["valorTotalHomologado"]
            .sum()
        )

    df_mapa["valor"] = df_mapa["uf"].map(valor_uf_periodo).fillna(0)
    df_mapa["valor_fmt"] = df_mapa["valor"].apply(formatar_moeda)

    titulo_legenda = (
        f"Quantidade de contratações em {periodo}"
        if periodo != "Total"
        else "Quantidade de contratações - total 2020 a 2026"
    )

    # tiles="OpenStreetMap" -> mapa-base gratuito, sem exigir chave de API
    m = folium.Map(location=[-14.2, -51.9], zoom_start=4, tiles="OpenStreetMap")

    folium.Choropleth(
        geo_data=geojson_uf,
        data=df_mapa,
        columns=["uf", "quantidade"],
        key_on="feature.properties.uf",
        fill_color="YlOrRd",
        fill_opacity=0.85,
        line_opacity=0.4,
        nan_fill_color="white",
        legend_name=titulo_legenda,
    ).add_to(m)

    # Adiciona quantidade e valor ao GeoJSON para aparecer no tooltip (ao passar o mouse)
    valores_quantidade = df_mapa.set_index("uf")["quantidade"].to_dict()
    valores_valor_fmt = df_mapa.set_index("uf")["valor_fmt"].to_dict()

    for feature in geojson_uf["features"]:
        sigla = feature["properties"]["uf"]
        feature["properties"]["quantidade"] = int(valores_quantidade.get(sigla, 0))
        feature["properties"]["valor_fmt"] = valores_valor_fmt.get(sigla, formatar_moeda(0))

    folium.GeoJson(
        geojson_uf,
        style_function=lambda x: {"fillOpacity": 0, "weight": 0},
        tooltip=folium.GeoJsonTooltip(
            fields=["uf", "quantidade", "valor_fmt"],
            aliases=["UF:", "Contratações:", "Valor homologado:"],
        ),
    ).add_to(m)

    return m


def resumo_periodo(periodo):
    quantidade = int(quantidade_por_periodo.loc[periodo])
    valor = valor_por_periodo.loc[periodo]
    principais_areas = top_areas_periodo(periodo, top_n=2)

    linhas_areas = "\n".join(
        f"{i + 1}. **{area}** — {qtd} contratações"
        for i, (area, qtd) in enumerate(principais_areas.items())
    )

    titulo_periodo = f"ano de {periodo}" if periodo != "Total" else "período completo (2020 a 2026)"

    texto = (
        f"**Resumo — {titulo_periodo}**\n\n"
        f"- Quantidade de contratações: **{quantidade}**\n"
        f"- Valor total homologado: **{formatar_moeda(valor)}**\n"
        f"- Áreas mais contratadas:\n{linhas_areas}\n"
    )
    return Markdown(texto)


def mostrar_mapa(periodo):
    display(criar_mapa(periodo))
    display(resumo_periodo(periodo))


opcoes_periodo = list(contagem_ano_uf.index)

interact(
    mostrar_mapa,
    periodo=Dropdown(options=opcoes_periodo, value="Total", description="Período:"),
);


interactive(children=(Dropdown(description='Período:', index=6, options=(2021, 2022, 2023, 2024, 2025, 2026, '…

### Salvando o mapa do período total em HTML


In [22]:
criar_mapa("Total").save("mapa_contratacoes_uf_total.html")
print("Mapa salvo em mapa_contratacoes_uf_total.html")


Mapa salvo em mapa_contratacoes_uf_total.html


## Agrupamento regional, tabelas do artigo e snapshot dos dados

Células usadas no artigo: regiões do IBGE, tabelas por ano e por região, período real coberto em cada ano, UF subrogada/esfera e arquivos para o repositório.

In [23]:
regiao = {**dict.fromkeys(["SP","RJ","MG","ES"], "Sudeste"),
          **dict.fromkeys(["MA","PI","CE","RN","PB","PE","AL","SE","BA"], "Nordeste"),
          **dict.fromkeys(["PR","SC","RS"], "Sul"),
          **dict.fromkeys(["MT","MS","GO","DF"], "Centro-Oeste"),
          **dict.fromkeys(["AC","AP","AM","PA","RO","RR","TO"], "Norte")}
df_limpo["regiao"] = df_limpo["uf"].map(regiao)

tabela_regiao = df_limpo.groupby("regiao").agg(
    registros=("idCompra", "size"),
    com_valor=("valorTotalHomologado", "count"),
    valor_total=("valorTotalHomologado", "sum"),
)
tabela_regiao["pct_registros"] = (tabela_regiao["registros"] / tabela_regiao["registros"].sum() * 100).round(2)
tabela_regiao["pct_valor"] = (tabela_regiao["valor_total"] / tabela_regiao["valor_total"].sum() * 100).round(2)
tabela_regiao = tabela_regiao.sort_values("registros", ascending=False)
tabela_regiao

,registros,com_valor,valor_total,pct_registros,pct_valor
regiao,,,,,
Sudeste,2633,2016,1.009308e+09,45.57,43.03
Nordeste,936,726,9.664905e+07,16.20,4.12
Sul,857,608,5.985200e+07,14.83,2.55
Centro-Oeste,803,586,1.089063e+09,13.90,46.43
Norte,549,338,9.059398e+07,9.50,3.86


In [24]:
# Tabela por ano (após tratamento) e período real coberto por cada ano
tabela_ano = df_limpo.groupby("anoConsulta").agg(
    registros=("idCompra", "size"),
    valor_total=("valorTotalHomologado", "sum"),
)
print(tabela_ano)
print()
print(df_limpo.groupby("anoConsulta")["dataPublicacaoPncp"].agg(["min", "max", "count"]))

             registros   valor_total
anoConsulta                         
2021               813  2.987554e+06
2022               968  3.474786e+07
2023               999  4.819370e+07
2024               998  2.633323e+08
2025              1000  2.591025e+08
2026              1000  1.737102e+09

                            min                 max  count
anoConsulta                                               
2021        2021-08-10 07:54:24 2021-09-14 07:01:00    813
2022        2022-01-03 07:00:00 2022-02-11 15:16:18    968
2023        2023-01-02 08:44:55 2023-01-25 12:23:54    999
2024        2024-01-02 08:59:18 2024-01-17 16:33:35    998
2025        2025-01-02 08:29:46 2025-01-10 12:25:08   1000
2026        2026-01-02 08:43:00 2026-01-12 16:00:05   1000


In [25]:
# UF da unidade x UF subrogada e esfera administrativa (base bruta)
sub = df_infos["unidadeSubrogadaUfSigla"]
print("Com UF subrogada:", sub.notna().mean())
print("UF subrogada diferente da UF do órgão:", (sub.notna() & (sub != df_infos["unidadeOrgaoUfSigla"])).mean())
print(df_infos["orgaoEntidadeEsferaId"].value_counts(normalize=True))

Com UF subrogada: 0.0005
UF subrogada diferente da UF do órgão: 0.0
orgaoEntidadeEsferaId
F    0.636833
E    0.259167
M    0.081333
N    0.015667
D    0.007000
Name: proportion, dtype: float64


In [26]:
import os
os.makedirs("dados", exist_ok=True)
os.makedirs("resultados", exist_ok=True)

df_limpo.to_csv("dados/contratacoes_dispensa_snapshot.csv", index=False)
tabela_ano.to_csv("resultados/tabela_ano.csv")
stats_valor_uf.assign(quantidade_contratacoes=freq_uf).sort_values("quantidade_contratacoes", ascending=False).to_csv("resultados/tabela_uf.csv")
tabela_regiao.to_csv("resultados/tabela_regiao.csv")
print("Arquivos salvos.")

Arquivos salvos.


## Diagnóstico para a revisão do artigo
Preenche os campos marcados em amarelo no artigo.

In [27]:
# ===== DIAGNÓSTICO PARA A REVISÃO (rodar no kernel onde df_infos, df e df_limpo existem) =====
import sys, os, importlib.metadata as md
import pandas as pd
os.makedirs("resultados", exist_ok=True)

# 1) VERSÕES
pacotes = ["requests", "pandas", "matplotlib", "folium", "ipywidgets"]
linhas = [f"Python {sys.version.split()[0]}"]
for p in pacotes:
    try: linhas.append(f"{p} {md.version(p)}")
    except Exception: pass
open("resultados/versoes.txt", "w").write("\n".join(linhas))
print("VERSOES:", "; ".join(linhas))

raw = df_infos.copy()
raw["uf"] = raw["unidadeOrgaoUfSigla"]

# 2) ESFERA (F=federal, E=estadual, M=municipal)
print("\nESFERA (base bruta, 6.000):\n", (raw["orgaoEntidadeEsferaId"].value_counts(normalize=True)*100).round(2))
dedup = raw.drop_duplicates(subset="idCompra")
print("ESFERA (após remover duplicados, 5.572):\n", (dedup["orgaoEntidadeEsferaId"].value_counts(normalize=True)*100).round(2))

# 3) DUPLICIDADES
dup_ids = raw.loc[raw["idCompra"].duplicated(keep=False), "idCompra"].unique()
d = raw[raw["idCompra"].isin(dup_ids)]
anos_por_id = d.groupby("idCompra")["anoConsulta"].nunique()
print("\nDUPLICIDADES: ids repetidos =", len(dup_ids), "| removidos =", raw["idCompra"].duplicated().sum())
print("  ids que aparecem em mais de um ano de consulta:", int((anos_por_id > 1).sum()))
print("  ids repetidos dentro do mesmo ano de consulta:", int((anos_por_id == 1).sum()))
print("  linhas idênticas em todas as 52 colunas:", int(raw.astype(str).duplicated().sum()))
dif = {}
for _, g in d.groupby("idCompra"):
    for c in raw.columns:
        if g[c].astype(str).nunique() > 1: dif[c] = dif.get(c, 0) + 1
print("  colunas que mais diferem entre repetições:", sorted(dif.items(), key=lambda x: -x[1])[:8])

# 4) AUSÊNCIA DE VALOR HOMOLOGADO POR UF (base bruta)
aus = (raw.assign(sem_valor=raw["valorTotalHomologado"].isna())
          .groupby("uf")["sem_valor"].agg(registros="size", sem_valor="sum"))
aus["pct_sem_valor"] = (aus["sem_valor"] / aus["registros"] * 100).round(2)
aus = aus.sort_values("pct_sem_valor", ascending=False)
aus.to_csv("resultados/ausencia_valor_por_uf.csv")
print("\nTOTAL sem valor:", int(aus["sem_valor"].sum()), "de", int(aus["registros"].sum()))
print("MAIORES % sem valor:\n", aus.head(5)); print("MENORES % sem valor:\n", aus.tail(5))

# 5) REGISTRO MÁXIMO
i = raw["valorTotalHomologado"].idxmax()
cols = ["idCompra","uf","orgaoEntidadeRazaoSocial","unidadeOrgaoNomeUnidade","objetoCompra",
        "valorTotalEstimado","valorTotalHomologado","anoConsulta","dataPublicacaoPncp","situacaoCompraNomePncp"]
print("\nREGISTRO MÁXIMO:"); print(raw.loc[i, cols].to_string())
print("  homologado/estimado =", raw.loc[i,"valorTotalHomologado"] / raw.loc[i,"valorTotalEstimado"])
print("  repetido na base bruta?", int((raw["idCompra"] == raw.loc[i,"idCompra"]).sum()) > 1)


VERSOES: Python 3.14.7; requests 2.34.2; pandas 3.1.0; matplotlib 3.11.2; folium 0.20.0; ipywidgets 8.1.9

ESFERA (base bruta, 6.000):
 orgaoEntidadeEsferaId
F    63.68
E    25.92
M     8.13
N     1.57
D     0.70
Name: proportion, dtype: float64
ESFERA (após remover duplicados, 5.572):
 orgaoEntidadeEsferaId
F    62.53
E    26.72
M     8.45
N     1.63
D     0.67
Name: proportion, dtype: float64

DUPLICIDADES: ids repetidos = 193 | removidos = 222
  ids que aparecem em mais de um ano de consulta: 0
  ids repetidos dentro do mesmo ano de consulta: 193
  linhas idênticas em todas as 52 colunas: 0
  colunas que mais diferem entre repetições: [('numeroControlePNCP', 193), ('sequencialCompraPncp', 193), ('dataInclusaoPncp', 193), ('dataAtualizacaoPncp', 193), ('dataPublicacaoPncp', 193), ('contratacaoExcluida', 188), ('existeResultado', 95), ('dataAberturaPropostaPncp', 3)]

TOTAL sem valor: 1630 de 6000
MAIORES % sem valor:
     registros  sem_valor  pct_sem_valor
uf                        